# Topic: Agentic Reasoning Patterns (ReAct, Plan-and-Solve, Reflection, Tree of Thoughts)

## Definition (30-second explanation)
*   **Naive Prompting** is asking a junior chef to cook a 5-course meal in one go and hoping it works. 
*   **ReAct (Reasoning + Acting)** is the chef deciding to make soup (Thought), adding salt (Action), tasting it (Observation), and realizing it needs pepper (Thought).
*   **Plan-and-Solve** is writing out the exact recipe for all 5 courses before turning on the stove. 
*   **Reflection (Self-Correction)** is having a head chef taste the dish, point out it's too salty, and making the junior chef remake it before serving.
*   **Tree of Thoughts (ToT)** is the kitchen testing 3 completely different recipes simultaneously, evaluating which tastes best at each step, and throwing away the bad ones.

## Why Interviewers Ask This
*   LLMs are fundamentally autoregressive next-token predictors; they cannot "think ahead." 
*   Interviewers want to see if you know how to use control flow (Python loops, graphs) and specific prompting frameworks to force the LLM to plan, verify its own work, and recover from inevitable API or logic errors.

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** In complex, multi-step tasks (like querying a database and joining tables), a single zero-shot LLM call almost always hallucinates or fails because it lacks the ability to test assumptions or fix syntax errors on the fly.
*   **The Mechanism:**
    *   **ReAct:** A strictly enforced prompt loop: `Thought` (what should I do?) $\rightarrow$ `Action` (call a tool) $\rightarrow$ `Observation` (tool output). Repeats until the task is solved.
    *   **Plan-and-Solve:** A two-agent system. Agent 1 generates a DAG (Directed Acyclic Graph) of steps. Agent 2 executes them sequentially. 
    *   **Reflection / Critic:** Agent 1 generates an answer/code. Agent 2 (Critic) reviews it against the original prompt and outputs a critique. Agent 1 regenerates based on the critique.
    *   **Tree of Thoughts:** Explores multiple reasoning paths (branches), uses an LLM to score the viability of each path, and uses algorithms like BFS/DFS to traverse to the best final answer.
*   **The Trade-off:** Every advanced reasoning pattern trades **Compute/Latency** for **Accuracy**. ReAct requires multiple sequential API calls (slow). Tree of Thoughts requires exponential API calls (prohibitively expensive for real-time production).

## When to Use
*   **ReAct:** Dynamic environments where the next step depends entirely on the result of the previous step (e.g., Web browsing, API exploration).
*   **Plan-and-Solve:** Highly structured, deterministic tasks with known sub-steps (e.g., data pipeline orchestration).
*   **Reflection:** Code generation, Text-to-SQL workflows, and highly formatted document generation.
*   **Tree of Thoughts:** Complex logical puzzles, mathematical proofs, or offline strategic planning where latency does not matter.

## Advantages
*   **ReAct:** Highly autonomous and adaptable to unexpected tool outputs.
*   **Reflection:** Drastically improves coding/SQL accuracy by treating the compiler/database engine errors as feedback.
*   **Plan-and-Solve:** Cheaper and faster than ReAct because it doesn't need to re-evaluate the entire state after every single tiny action.

## Limitations
*   **Infinite Loops:** ReAct agents frequently get stuck calling the same failing tool over and over if not capped by a `max_iterations` limit.
*   **Context Window Bloat:** Keeping the entire `Thought/Action/Observation` history in the prompt quickly consumes the 128k/200k token limits and dilutes attention.
*   **Brittleness (Plan-and-Solve):** If Step 1 of the plan fails, Steps 2-5 are completely useless, requiring a full replan.

## Common Comparisons
*   **ReAct vs. Plan-and-Solve:** ReAct interleaves thinking and acting (agile but slow/expensive). Plan-and-Solve separates planning from execution (waterfall, faster, but fragile if a step fails).
*   **Chain of Thought (CoT) vs. ReAct:** CoT is just asking the model to "think step by step" in text. ReAct actually pauses generation to execute external tools and inject real-world observations.

## Common Interview Traps
*   **Proposing Tree of Thoughts for a Chatbot:** Recommending ToT for a real-time customer-facing app shows a lack of production experience. ToT takes minutes and costs dollars per query.
*   **Confusing ReAct with React.js:** ReAct stands for "Reasoning and Acting." It has absolutely nothing to do with the frontend web framework.

## Python Syntax (LangChain ReAct Agent Minimal Implementation)
```python
from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(model="gpt-4o")
tools = [my_sql_tool, my_weather_tool]

# 1. The ReAct Prompt forces the LLM to output "Thought:", "Action:", "Observation:"
prompt = PromptTemplate.from_template("""
Answer the question using the following tools: {tools}
Use this exact format:
Question: the input
Thought: what you should do
Action: the tool to use
Action Input: tool parameters
Observation: tool result
... (Thought/Action/Observation can repeat N times)
Final Answer: the output
""")

# 2. Bind LLM, Tools, and Prompt into the ReAct logic
agent = create_react_agent(llm, tools, prompt)

# 3. AgentExecutor handles the actual while loop and handles max_iterations
agent_executor = AgentExecutor(agent=agent, tools=tools, max_iterations=5, verbose=True)

# 4. Run the loop
result = agent_executor.invoke({"input": "What were our Q3 sales?"})
```

## 45-Second Interview Answer
"Standard LLM prompting fails on complex tasks because models cannot plan ahead or recover from errors. In production, I rely on Agentic Reasoning patterns. For dynamic tasks, I use **ReAct**, which forces the LLM into a loop of Reasoning, Acting via tools, and observing the results. For deterministic tasks, I prefer **Plan-and-Solve** to reduce latency by outlining all steps upfront. My most heavily utilized pattern is the **Self-Correction/Reflection loop**, especially for code and SQL generation: an Actor agent writes the code, executes it, and if it fails, a Critic agent reads the traceback and patches the code. I generally avoid **Tree of Thoughts** in real-time systems due to its exponential latency and token costs."